In [1]:
###############################################################################
# Open up the connection to pangeo
###############################################################################
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import dask
from dask.diagnostics import progress
from tqdm.autonotebook import tqdm
import intake
import fsspec
import seaborn as sns
import sys
from collections import defaultdict
import nc_time_axis
import fsspec
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 12, 6
# open up the full pangeo set of data
pangeo = intake.open_esm_datastore("https://storage.googleapis.com/cmip6/pangeo-cmip6.json")


x = pangeo.df.loc[(pangeo.df['source_id'] == 'MPI-ESM1-2-LR') & (pangeo.df['variable_id'] == 'tas') & (pangeo.df['table_id'] == 'Amon')].copy()



/Users/snyd535/miniconda3/envs/pangeo/lib/python3.7/site-packages/ipykernel_launcher.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  # Remove the CWD from sys.path while we load stuff.


In [10]:
# The MPI-LR Tgav values in our records have the years out of order.
# Check one example

# SSP245 is where we see that data from our Tgav calculations stops at 2039 for all but 3 realizations.
# Is it our Tgav code or the data? Check the data for one of the realizations that stops in 2039 in 
# our files.
x1 = x[x['experiment_id'] == 'ssp585'].copy()
x1 = x1[x1['member_id'] == 'r1i1p1f1'].copy()


print(x1.zstore.values[0])

gs://cmip6/CMIP6/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp585/r1i1p1f1/Amon/tas/gn/v20190710/


In [11]:
# open up that file and take a look what years of data are actually included in the netcdf
ds = xr.open_zarr(fsspec.get_mapper(x1.zstore.values[0]), consolidated=True)
ds

<xarray.Dataset>
Dimensions:    (bnds: 2, lat: 96, lon: 192, time: 1032)
Coordinates:
    height     float64 ...
  * lat        (lat) float64 -88.57 -86.72 -84.86 -83.0 ... 84.86 86.72 88.57
    lat_bnds   (lat, bnds) float64 dask.array<chunksize=(96, 2), meta=np.ndarray>
  * lon        (lon) float64 0.0 1.875 3.75 5.625 ... 352.5 354.4 356.2 358.1
    lon_bnds   (lon, bnds) float64 dask.array<chunksize=(192, 2), meta=np.ndarray>
  * time       (time) datetime64[ns] 2035-01-16T12:00:00 ... 2034-12-16T12:00:00
    time_bnds  (time, bnds) datetime64[ns] dask.array<chunksize=(1032, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    tas        (time, lat, lon) float32 dask.array<chunksize=(516, 96, 192), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   60265.0
    branch_time_in_parent:  60265.0
    cmor_version:           3.5.0
    contact:                cmip6-mpi-esm@dkrz.de
    creation_date:          2019-11-11T15:53:54Z
    data_specs_version:     01.00.30
    experiment:             update of RCP8.5 based on SSP5
    experiment_id:          ssp585
    external_variables:     areacella
    forcing_index:          1
    frequency:              mon
    further_info_url:       https://furtherinfo.es-doc.org/CMIP6.MPI-M.MPI-ES...
    grid:                   gn
    grid_label:             gn
    history:                2019-11-11T15:53:54Z ; CMOR rewrote data to be co...
    initialization_index:   1
    institution:            Max Planck Institute for Meteorology, Hamburg 201...
    institution_id:         MPI-M
    license:                CMIP6 model data produced by MPI-M is licensed un...
    mip_era:                CMIP6
    nominal_resolution:     250 km
    parent_activity_id:     CMIP
    parent_experiment_id:   historical
    parent_mip_era:         CMIP6
    parent_source_id:       MPI-ESM1-2-LR
    parent_time_units:      days since 1850-1-1 00:00:00
    parent_variant_label:   r1i1p1f1
    physics_index:          1
    product:                model-output
    project_id:             CMIP6
    realization_index:      1
    realm:                  atmos
    references:             MPI-ESM: Mauritsen, T. et al. (2019), Development...
    source:                 MPI-ESM1.2-LR (2017): \naerosol: none, prescribed...
    source_id:              MPI-ESM1-2-LR
    source_type:            AOGCM
    status:                 2020-03-22;created; by gcs.cmip6.ldeo@gmail.com
    sub_experiment:         none
    sub_experiment_id:      none
    table_id:               Amon
    table_info:             Creation Date:(09 May 2019) MD5:e6ef8ececc8f33864...
    title:                  MPI-ESM1-2-LR output prepared for CMIP6
    tracking_id:            hdl:21.14100/6e450195-bd31-4ffa-9887-479288eaea90...
    variable_id:            tas
    variant_label:          r1i1p1f1
    netcdf_tracking_ids:    hdl:21.14100/6e450195-bd31-4ffa-9887-479288eaea90...
    version_id:             v20190710

In [12]:
np.set_printoptions(threshold=sys.maxsize) # no truncation to check if 
# netcdf has years out of order

print(ds.time.values)

['2035-01-16T12:00:00.000000000' '2035-02-15T00:00:00.000000000'
 '2035-03-16T12:00:00.000000000' '2035-04-16T00:00:00.000000000'
 '2035-05-16T12:00:00.000000000' '2035-06-16T00:00:00.000000000'
 '2035-07-16T12:00:00.000000000' '2035-08-16T12:00:00.000000000'
 '2035-09-16T00:00:00.000000000' '2035-10-16T12:00:00.000000000'
 '2035-11-16T00:00:00.000000000' '2035-12-16T12:00:00.000000000'
 '2036-01-16T12:00:00.000000000' '2036-02-15T12:00:00.000000000'
 '2036-03-16T12:00:00.000000000' '2036-04-16T00:00:00.000000000'
 '2036-05-16T12:00:00.000000000' '2036-06-16T00:00:00.000000000'
 '2036-07-16T12:00:00.000000000' '2036-08-16T12:00:00.000000000'
 '2036-09-16T00:00:00.000000000' '2036-10-16T12:00:00.000000000'
 '2036-11-16T00:00:00.000000000' '2036-12-16T12:00:00.000000000'
 '2037-01-16T12:00:00.000000000' '2037-02-15T00:00:00.000000000'
 '2037-03-16T12:00:00.000000000' '2037-04-16T00:00:00.000000000'
 '2037-05-16T12:00:00.000000000' '2037-06-16T00:00:00.000000000'
 '2037-07-16T12:00:00.000

In [ ]:
np.set_printoptions(threshold=False) # reset so truncation occurs